In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="KTH/hungarian-single-speaker-tts", 
                  repo_type="dataset", local_dir="./hungarian-single-speaker-tts")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 9 files: 100%|██████████| 9/9 [00:02<00:00,  3.22it/s]


'/home/ubuntu/hungarian-single-speaker-tts'

In [3]:
files = glob('hungarian-single-speaker-tts/data/*.parquet')
files

['hungarian-single-speaker-tts/data/train-00002-of-00007-a1adbbd862c212d9.parquet',
 'hungarian-single-speaker-tts/data/train-00006-of-00007-a9a5f792d4b99940.parquet',
 'hungarian-single-speaker-tts/data/train-00003-of-00007-aa763c268da02b51.parquet',
 'hungarian-single-speaker-tts/data/train-00000-of-00007-e7de4bf2d1b4e37e.parquet',
 'hungarian-single-speaker-tts/data/train-00001-of-00007-a03d6983ca39134e.parquet',
 'hungarian-single-speaker-tts/data/train-00004-of-00007-7961133cdf812f7a.parquet',
 'hungarian-single-speaker-tts/data/train-00005-of-00007-5a8e7af193e57a28.parquet']

In [4]:
df = pd.read_parquet(files[0])
df

,id,audio,original_text,text,duration
0,egri_csillagok_1290,{'bytes': b'RIFF\xc2\xa5\x06\x00WAVEfmt \x12\x...,"Egy méltóságos tekintetű, szürke szakállú pasa...",egy méltóságos tekintetű szürke szakállú pasa ...,4.94
1,egri_csillagok_1291,{'bytes': b'RIFF\x02\xc0\x0b\x00WAVEfmt \x12\x...,Előtte hét lófarkas zászlót vittek. A fején re...,előtte hét lófarkas zászlót vittek a fején ren...,8.73
2,egri_csillagok_1292,{'bytes': b'RIFF\xd2\xe5\x0b\x00WAVEfmt \x12\x...,Lehetetlen! - De bizony. Az imént ment el a má...,lehetetlen de bizony az imént ment el a másik ...,8.84
3,egri_csillagok_1293,{'bytes': b'RIFF\xf2\x94\n\x00WAVEfmt \x12\x00...,Csúfnév - felelte Tulipán. S egy fűszálat szak...,csúfnév felelte tulipán s egy fűszálat szakíto...,7.86
4,egri_csillagok_1294,{'bytes': b'RIFF\xb24\n\x00WAVEfmt \x12\x00\x0...,"Egy csapat ezüst- és aranybuzogányos, ijesztőe...",egy csapat ezüst- és aranybuzogányos ijesztően...,7.58
...,...,...,...,...,...
640,egri_csillagok_1930,{'bytes': b'RIFF\xe2i\t\x00WAVEfmt \x12\x00\x0...,felelte Cecey. A pap tekintete magyarázatot ké...,felelte cecey a pap tekintete magyarázatot kér...,6.99
641,egri_csillagok_1931,{'bytes': b'RIFFb\xf5\x0c\x00WAVEfmt \x12\x00\...,A királyné igen megszerette. Nem ereszti. - Mi...,a királyné igen megszerette nem ereszti mióta ...,9.63
642,egri_csillagok_1932,{'bytes': b'RIFF2c\x0b\x00WAVEfmt \x12\x00\x00...,hörkent föl a pap. - Mellette van - felelte Ce...,hörkent föl a pap mellette van felelte cecey c...,8.46
643,egri_csillagok_1933,{'bytes': b'RIFF\xa2\x9d\x0b\x00WAVEfmt \x12\x...,"Vica csak épp hogy ott van, no. - A te leányod...",vica csak épp hogy ott van no a te leányod a s...,8.63


In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['original_text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"hungarian-single-speaker-tts"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 645/645 [00:43<00:00, 14.87it/s]


In [7]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'hungarian-single-speaker-tts_audio/hungarian-single-speaker-tts-data-train-00002-of-00007-a1adbbd862c212d9_0.mp3',
 'text': 'Egy méltóságos tekintetű, szürke szakállú pasa lötyögött a szultánfiak után.',
 'speaker': 'hungarian-single-speaker-tts'}

In [8]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'hungarian-single-speaker-tts')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 296.10ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  428kB /  428kB, 1.38MB/s  
Processing Files (1 / 1): 100%|██████████|  428kB /  428kB, 1.07MB/s  
New Data Upload: 100%|██████████|  428kB /  428kB, 1.07MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.41 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/82a0874c8d6023fb97f2888d77fad2988bfd95e4', commit_message='Upload dataset', commit_description='', oid='82a0874c8d6023fb97f2888d77fad2988bfd95e4', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('hungarian-single-speaker-tts-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [11]:
folders = glob('hungarian-single-speaker-tts_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

hungarian-single-speaker-tts_audio_neucodec
hungarian-single-speaker-tts_audio


In [12]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('hungarian-single-speaker-tts_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  15%|█▌        | 39.3MB /  254MB,   ???B/s  
Processing Files (0 / 1):  68%|██████▊   |  172MB /  254MB,  668MB/s  
Processing Files (0 / 1):  99%|█████████▉|  252MB /  254MB,  533MB/s  
Processing Files (0 / 1): 100%|█████████▉|  253MB /  254MB,  267MB/s  
Processing Files (0 / 1): 100%|█████████▉|  254MB /  254MB,  214MB/s  
Processing Files (1 / 1): 100%|██████████|  254MB /  254MB,  179MB/s  
Processing Files (1 / 1): 100%|██████████|  254MB /  254MB,  153MB/s  
New Data Upload: 100%|██████████|  254MB /  254MB,  153MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 7.07MB / 7.07MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 7.07MB / 7.07MB,  0.00B/s  
New Data Upload: 100%|██████████| 7.07MB / 7.07MB,  0.00B/s  
